In [ ]:
%load_ext autoreload
%autoreload 2

# 02 — Transform

Unify all regional CSVs from `data/01_raw/` into a single standardised dataset at `data/02_transformed/unified_crime_data.csv`.

## Run the Unification Pipeline

In [ ]:
from gta_crime_data.transform.unify_datasets import run

run()

## Inspect the Unified Dataset

In [ ]:
import os
import pandas as pd

unified_path = os.path.join('..', 'data', '02_transformed', 'unified_crime_data.csv')
df = pd.read_csv(unified_path, low_memory=False)

print(f'Shape: {len(df):,} rows × {len(df.columns)} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()

# Date range
df['occurrence_date'] = pd.to_datetime(df['occurrence_date'], errors='coerce')
print(f'Date range: {df["occurrence_date"].min().date()} → {df["occurrence_date"].max().date()}')
print()

# Nulls
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls):
    print('Missing values:')
    for col, n in nulls.items():
        print(f'  {col:25s} {n:>8,}  ({n/len(df)*100:.1f}%)')
else:
    print('No missing values.')
print()

# Unique counts
print('Unique values per column:')
for col in df.columns:
    print(f'  {col:25s} {df[col].nunique():>8,}')


## Filter Invalid Data and Validate Schema
We'll separate rows that are missing required values into an `invalid_data.csv` file, and ensure the remaining valid dataset adheres strictly to our `unified_schema`.

In [ ]:
import pandera.errors as pa_errors
from gta_crime_data.transform.schemas import unified_schema

# Use lazy=True to catch all schema errors without halting on the first one
try:
    unified_schema.validate(df, lazy=True)
    valid_df = df.copy()
    invalid_df = pd.DataFrame(columns=df.columns)
    print("All rows are valid.")
except pa_errors.SchemaErrors as err:
    print(f"Found {len(err.failure_cases)} validation errors affecting {err.failure_cases['index'].nunique()} rows.")
    
    invalid_indices = err.failure_cases['index'].unique()
    
    # Separate invalid and valid data
    invalid_df = df.loc[invalid_indices]
    valid_df = df.drop(index=invalid_indices)
    
    invalid_path = os.path.join('..', 'data', '02_transformed', 'invalid_data.csv')
    invalid_df.to_csv(invalid_path, index=False)
    print(f"Saved {len(invalid_df):,} invalid rows to {invalid_path}")


In [ ]:
# Validate the remaining dataset to ensure no invalid rows slipped through
unified_schema.validate(valid_df)
print(f"Success! The remaining {len(valid_df):,} rows passed validation.")

# Overwrite the unified data with only the valid records
valid_df.to_csv(unified_path, index=False)
print(f"Overwrote {unified_path} with valid dataset.")

# Re-assign back to 'df' for the rest of the notebook exploration
df = valid_df.copy()


### De-duplicate by source_identifier, merging multi-offence rows

In [ ]:
from gta_crime_data.transform.deduplicate_incidents import deduplicate_incidents

df_deduped = deduplicate_incidents(df)
df_deduped

In [ ]:
pre_2010 = df[df['occurrence_date'] < '2010-01-01']
print(f'Rows before 2010: {len(pre_2010):,} ({len(pre_2010)/len(df)*100:.2f}%)')
print()

# Breakdown by region and year
pre_2010['year'] = pre_2010['occurrence_date'].dt.year
print(pre_2010.groupby(['region', 'year']).size().to_string())
print()

# Which raw files contain the pre-2010 records?
print('Pre-2010 records by source file:')
print(pre_2010.groupby('source_file_name').size().to_string())
print()

# Example trace
sample = pre_2010.iloc[0]
print(f'Sample:')
print(f'  → Open: data/01_raw/{sample["source_file_name"]}.csv')
print(f'  → Find ID: {sample["source_identifier"]}')


In [ ]:
df.info()

## Region Breakdown

In [ ]:
df['region'].value_counts()

## Crime Category Breakdown

In [ ]:
df['mapped_crime_category'].value_counts()

### 'MULTIPLE' Categories by Region

In [ ]:
df_deduped[df_deduped['mapped_crime_category'] == 'MULTIPLE'].groupby('region').size().sort_values(ascending=False)